1.Imports and setup

In [0]:
pip install uuid


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import  uuid


In [0]:
spark.sql("use catalog novacart_adb")
spark.sql("create schema if not exists silver_schema")
silver_run__id=str(uuid.uuid4())
print("current silver run id is "+silver_run__id)



current silver run id is 7ae76765-e748-4d27-9ef0-f2ac2f93774a


2.Silver control table

In [0]:
spark.sql("""
          create table if not exists novacart_adb.silver_schema.processing_control(
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            upated_at timestamp
          )
          using delta""")

DataFrame[]

3.helper function

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
  if spark.catalog.tableExists(target_table):
    print("table exists")
    dt = DeltaTable.forName(spark, target_table)
    (dt.alias("target")
    .merge(df_source.alias("source"),f"target.{join_key}=source.{join_key}")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
  else:
    print("table does not exists")
    df_source.write.mode("overwrite").saveAsTable(target_table)


In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl=(
        spark.table("novacart_adb.silver_schema.processing_control")
        .filter(
            (F.col("layer")=="silver")&
            (F.col("entity_name")==entity_name)&
            (F.col("run_status")=="success")
        )
        .orderBy(F.col("upated_at").desc())
        .limit(1)
    )
    rows=ctrl.collect()
    if len(rows)==0:
        return None
    else:
        return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
def upsert_silver_control(entity_name,last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged):
    ctrl_df=spark.createDataFrame(
            [
                (
                    "silver",
                    entity_name,
                    last_processed_bronze_run_id,
                    last_processed_bronze_ingested_at,
                    int(rows_merged),
                    "success",
                    silver_run__id,
                    datetime.now()
                )
            ],
            schema="""
                layer string,
                entity_name string,
                last_processed_bronze_run_id string,
                last_processed_bronze_ingested_at timestamp,
                rows_merged bigint,
                run_status string,
                silver_run_id string,
                upated_at timestamp
            """
        )
    dt=DeltaTable.forName(spark,"novacart_adb.silver_schema.processing_control")
    (dt.alias("target")
    .merge(ctrl_df.alias("source"),"target.layer=source.layer and target.entity_name=source.entity_name")
    .whenMatchedUpdate(set={
        "last_processed_bronze_run_id":"source.last_processed_bronze_run_id",
        "last_processed_bronze_ingested_at":"source.last_processed_bronze_ingested_at",
        "rows_merged":"source.rows_merged",
        "run_status":"source.run_status",
        "silver_run_id":"source.silver_run_id",
        "upated_at":"source.upated_at"
    })
    .whenNotMatchedInsertAll()
    .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table: str, entity_name: str):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df, last_ingested_at
    return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)), last_ingested_at

4.ordersincremental processing


In [0]:
df_raw=spark.sql("select * from novacart_adb.bronze_schema.orders")
display(df_raw)


order_id,customer_id,product_id,order_status,order_amount,created_at,updated_at,bronze_ingested_at,bronze_run_id,bronze_source_table
200001,5001,1001,PLACED,51.00,2026-02-01T10:01:00.000Z,2026-02-01T10:01:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200002,5002,1002,PLACED,52.00,2026-02-01T10:02:00.000Z,2026-02-01T10:02:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200003,5003,1003,PLACED,53.00,2026-02-01T10:03:00.000Z,2026-02-01T10:03:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200004,5004,1004,PLACED,54.00,2026-02-01T10:04:00.000Z,2026-02-01T10:04:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200005,5005,1005,PLACED,55.00,2026-02-01T10:05:00.000Z,2026-02-01T10:05:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200006,5006,1006,PLACED,56.00,2026-02-01T10:06:00.000Z,2026-02-01T10:06:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200007,5007,1007,PLACED,57.00,2026-02-01T10:07:00.000Z,2026-02-01T10:07:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200008,5008,1008,PLACED,58.00,2026-02-01T10:08:00.000Z,2026-02-01T10:08:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200009,5009,1009,cancelled,59.00,2026-02-01T10:09:00.000Z,2026-02-01T10:09:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders
200010,5010,1010,PLACED,60.00,2026-02-01T10:10:00.000Z,2026-02-01T10:10:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders


In [0]:
orders_inc, last_orders_ingested_at = get_incremental_bronze("novacart_adb.bronze_schema.orders", "orders")
orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver={orders_inc_count}")
if orders_inc_count > 0:
    order_window = Window.partitionBy("order_id").orderBy(F.col("updated_at").cast("timestamp").desc(), F.col("bronze_ingested_at").desc())
    orders_cleaned = (
        orders_inc
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status", F.when(F.col("order_status") == "", F.lit("NONE")).otherwise(F.col("order_status")))
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$,]", ""))
        .withColumn("order_amount", F.when(F.col("order_amount").isin("N/A", "NULL", "?", ""), None).otherwise(F.col("order_amount")))
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        .withColumn("row_rank", F.row_number().over(order_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run__id))
    )
    upsert_to_silver(orders_cleaned, "novacart_adb.silver_schema.orders_cleaned", "order_id")
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            F.when(F.col("customer_id").isNull(), "verify customer_id")
            .when(F.col("product_id").isNull(), "verify product_id")
            .when(F.col("order_status").isNull(), "verify order_status")
            .when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), "verify order_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), F.lit(True)).otherwise(F.lit(False))
        )
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_month", F.month("created_at"))
        .withColumn("order_year", F.year("created_at"))
        .withColumn("order_day", F.dayofmonth("created_at"))
        .withColumn("order_dow", F.date_format("created_at", "E"))
    )
    orders_good = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No Issues")
    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )
    upsert_to_silver(orders_good, "novacart_adb.silver_schema.orders_transformed", "order_id")
    orders_bad.write.format("delta").mode("append").saveAsTable("novacart_adb.silver_schema.orders_quarantine")
    mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias('mx')).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(F.col("bronze_ingested_at") == mx_ingested)
        .agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    )
    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())
else:
    print("No new orders Bronze rows for silver.")
    upsert_silver_control(
        "orders",
        None,
        last_orders_ingested_at,
        orders_inc_count
    )

orders rows_to_process_in_silver=1500
table exists
table exists


In [0]:
spark.sql("ALTER TABLE novacart_adb.silver_schema.processing_control SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")

DataFrame[]

In [0]:
%sql
select * from novacart_adb.silver_schema.orders_cleaned;

order_id,customer_id,product_id,order_status,order_amount,created_at,updated_at,bronze_ingested_at,bronze_run_id,bronze_source_table,silver_run_id
200001,5001,1001,PLACED,51.0,2026-02-01T10:01:00.000Z,2026-02-01T10:01:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200002,5002,1002,PLACED,52.0,2026-02-01T10:02:00.000Z,2026-02-01T10:02:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200003,5003,1003,PLACED,53.0,2026-02-01T10:03:00.000Z,2026-02-01T10:03:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200004,5004,1004,PLACED,54.0,2026-02-01T10:04:00.000Z,2026-02-01T10:04:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200005,5005,1005,PLACED,55.0,2026-02-01T10:05:00.000Z,2026-02-01T10:05:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200006,5006,1006,PLACED,56.0,2026-02-01T10:06:00.000Z,2026-02-01T10:06:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200007,5007,1007,PLACED,57.0,2026-02-01T10:07:00.000Z,2026-02-01T10:07:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200008,5008,1008,PLACED,58.0,2026-02-01T10:08:00.000Z,2026-02-01T10:08:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200009,5009,1009,CANCELLED,59.0,2026-02-01T10:09:00.000Z,2026-02-01T10:09:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3
200010,5010,1010,PLACED,60.0,2026-02-01T10:10:00.000Z,2026-02-01T10:10:00.000Z,2026-03-30T19:08:17.990Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.orders,ad1a5606-4a65-4bc3-b4af-44ea9084d0f3


5.Products `incremental`processing

In [0]:
%sql
    
select * from novacart_adb.bronze_schema.products;

product_id,product_name,category,price,updated_at,bronze_ingested_at,bronze_run_id,bronze_source_table
1001,Product 1,Electronics,11.00,2026-02-01T09:01:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1002,Product 2,Electronics,12.00,2026-02-01T09:02:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1003,Product 3,Electronics,13.00,2026-02-01T09:03:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1004,Product 4,Electronics,14.00,2026-02-01T09:04:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1005,Product 5,Electronics,15.00,2026-02-01T09:05:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1006,Product 6,Electronics,16.00,2026-02-01T09:06:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1007,Product 7,electronics,17.00,2026-02-01T09:07:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1008,Product 8,Electronics,18.00,2026-02-01T09:08:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1009,Prod_9,Electronics,19.00,2026-02-01T09:09:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products
1010,Product 10,FITNESS,20.00,2026-02-01T09:10:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products


In [0]:
products_inc,last_orders_ingested_at=get_incremental_bronze("novacart_adb.bronze_schema.products","products")
products_inc_count=products_inc.count()
print(f"products rows_to_process_in_silver={products_inc_count}")
if products_inc_count > 0:
    product_window = Window.partitionBy("product_id").orderBy(F.col("updated_at").cast("timestamp").desc(), F.col("bronze_ingested_at").desc())
    products_cleaned = ( 
                        products_inc
                        .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))
                        .withColumn("product_name", F.when(F.col("product_name") == "", F.lit("NONE")).otherwise(F.col("product_name")))
                        .withColumn(
                            "category",
                            F.when(F.upper(F.trim(F.col("category"))).contains("ELECTRONICS"), "ELECTRONICS").otherwise(F.upper(F.trim(F.col("category"))))
                            )
                        .withColumn("price", F.trim(F.col("price")))
                        .withColumn("price", F.regexp_replace(F.col("price"), r"\$", ""))
                        .withColumn("price", F.regexp_replace(F.col("price"), ",","."))
                        .withColumn("price",F.regexp_replace(F.col("price"), r"\s+", ""))
                        .withColumn("price", F.expr("try_cast(price AS double)"))
                        .withColumn("updated_at", F.to_timestamp("updated_at"))
                        .withColumn("row_rank", F.row_number().over(product_window))
                        .filter(F.col("row_rank") == 1)
                        .drop("row_rank")
                        .withColumn("silver_run_id", F.lit(silver_run__id))
                        )
    upsert_to_silver(products_cleaned, "novacart_adb.silver_schema.products_cleaned", "product_id")
    products_validated = (
        products_cleaned
        .withColumn(
            "to_be_verified_by_products_team",
            F.when(F.col("product_name").isNull(), "verify product_name")
            .when(F.col("category").isNull(), "verify category")
            .when(F.col("price").isNull() | (F.col("price") <= 0), "verify price")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_product_price",
            F.when(F.col("price").isNull() | (F.col("price") <= 0),"Invalid_price").otherwise("Valid_price")
        )
    )
    products_good = products_validated.filter((F.col("to_be_verified_by_products_team") == "No Issues")&(F.col("check_product_price")=="Valid_price")
    )
    if"price_raw"in products_good.columns:
        products_good=products_good.drop("price_raw")

    products_bad = (
        products_validated
        .filter((F.col("to_be_verified_by_products_team") != "No Issues")|(F.col("check_product_price")!="Valid_price"))
        .withColumn("quarantine_ts",F.current_timestamp())
    )
    upsert_to_silver(products_good, "novacart_adb.silver_schema.products_transformed", "product_id")
    products_bad.write.format("delta").mode("append").saveAsTable("novacart_adb.silver_schema.products_quarantine")
    mx_ingested = products_inc.agg(F.max("bronze_ingested_at").alias('mx')).collect()[0]["mx"]
    mx_run = (
        products_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    )
    upsert_silver_control("products", mx_run, mx_ingested, products_good.count())
else:
    print("No new products Bronze rows for silver.")
    upsert_silver_control(
        "products",
        None,
        last_orders_ingested_at,
        products_inc_count
        
    )

products rows_to_process_in_silver=0
No new products Bronze rows for silver.


In [0]:
%sql
select * from novacart_adb.silver_schema.products_cleaned

product_id,product_name,category,price,updated_at,bronze_ingested_at,bronze_run_id,bronze_source_table,silver_run_id
1001,PRODUCT 1,ELECTRONICS,11.0,2026-02-01T09:01:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1002,PRODUCT 2,ELECTRONICS,12.0,2026-02-01T09:02:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1003,PRODUCT 3,ELECTRONICS,13.0,2026-02-01T09:03:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1004,PRODUCT 4,ELECTRONICS,14.0,2026-02-01T09:04:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1005,PRODUCT 5,ELECTRONICS,15.0,2026-02-01T09:05:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1006,PRODUCT 6,ELECTRONICS,16.0,2026-02-01T09:06:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1007,PRODUCT 7,ELECTRONICS,17.0,2026-02-01T09:07:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1008,PRODUCT 8,ELECTRONICS,18.0,2026-02-01T09:08:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1009,PROD_9,ELECTRONICS,19.0,2026-02-01T09:09:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6
1010,PRODUCT 10,FITNESS,20.0,2026-02-01T09:10:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6


In [0]:
%sql
select * from novacart_adb.silver_schema.products_quarantine

product_id,product_name,category,price,updated_at,bronze_ingested_at,bronze_run_id,bronze_source_table,silver_run_id,to_be_verified_by_products_team,check_product_price,quarantine_ts
1020,null,FITNESS,30.0,2026-02-01T09:20:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify product_name,Valid_price,2026-04-06T12:52:12.533Z
1025,PRODUCT 25,ELECTRONICS,null,2026-02-01T09:25:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify price,Invalid_price,2026-04-06T12:52:12.533Z
1040,null,FITNESS,50.0,2026-02-01T09:40:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify product_name,Valid_price,2026-04-06T12:52:12.533Z
1050,PRODUCT 50,FITNESS,null,2026-02-01T09:50:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify price,Invalid_price,2026-04-06T12:52:12.533Z
1060,null,FITNESS,70.0,2026-02-01T10:00:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify product_name,Valid_price,2026-04-06T12:52:12.533Z
1075,PRODUCT 75,ELECTRONICS,null,2026-02-01T10:15:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify price,Invalid_price,2026-04-06T12:52:12.533Z
1080,null,FITNESS,90.0,2026-02-01T10:20:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify product_name,Valid_price,2026-04-06T12:52:12.533Z
1100,null,FITNESS,null,2026-02-01T10:40:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify product_name,Invalid_price,2026-04-06T12:52:12.533Z
1120,null,FITNESS,40.0,2026-02-01T11:00:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,verify product_name,Valid_price,2026-04-06T12:52:12.533Z


6.Payments incremental processing

In [0]:
payments_inc,last_orders_ingested_at = get_incremental_bronze("novacart_adb.bronze_schema.payments","payments")
print("Payments last processed Bronze ingested_at =",last_orders_ingested_at)
payments_inc_count = payments_inc.count()
print(f"Payments last processed Bronze count ={payments_inc_count}")
if payments_inc_count > 0:
    payment_window = (
      Window.partitionBy("payment_id").orderBy(F.col("processed_at").desc(),F.col("bronze_ingested_at").desc())
    )
    payments_cleaned = (
        payments_inc
        .withColumn("payment_status",F.upper(F.col("payment_status")))
        .withColumn("payment_status",F.when(F.col("payment_status")=="",F.lit(None)).otherwise(F.col("payment_status")))
        .withColumn("paid_amount",F.trim(F.col("paid_amount")))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),r"\$",""))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),",","."))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),r"\s+",""))   
        .withColumn("paid_amount",F.expr("try_cast(paid_amount AS double)"))
        .withColumn("processed_at",F.to_timestamp(F.col("processed_at")))
        .withColumn("row_rank",F.row_number().over(payment_window))
        .filter(F.col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id",F.lit(silver_run__id))
    )
    upsert_to_silver(payments_cleaned,"novacart_adb.silver_schema.payments_cleaned","payment_id")
    payments_validated=(
        payments_cleaned
        .withColumn("to_be_verified_by_payments_team",F.when(F.col("order_id").isNull(), "verify_order_id")
            .when(F.col("payment_status").isNull(), "verify payment status")
            .when(F.col("paid_amount").isNull()|(F.col("paid_amount") <= 0), "verify paid amount")
            .otherwise("No Issues"))
        .withColumn("check_paid_amount",F.when(F.col("paid_amount").isNull()|(F.col("paid_amount") <= 0),F.lit("True")).otherwise("False"))
    )
    payments_good = (
        payments_validated
        .filter((F.col("to_be_verified_by_payments_team") == "No Issues"))
    )
    payments_bad = (
        payments_validated
        .filter((F.col("to_be_verified_by_payments_team") != "No Issues"))
        .withColumn("quarantine_ts",F.current_timestamp())
    )
    upsert_to_silver(payments_good, "novacart_adb.silver_schema.payments_transformed", "payment_id")
    payments_bad.write.format("delta").mode("append").saveAsTable("novacart_adb.silver_schema.payments_quarantine")
    mx_ingested = payments_inc.agg(F.max("bronze_ingested_at").alias('mx')).collect()[0]["mx"]
    mx_run=payments_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    upsert_silver_control("payments", mx_run, mx_ingested, payments_good.count())
else:
    print("No new payments Bronze rows for silver.")
    upsert_silver_control(
        "payments",
        None,
        last_orders_ingested_at,
        payments_inc_count
    )

Payments last processed Bronze ingested_at = 2026-03-30 19:08:32.248295
Payments last processed Bronze count =0
No new payments Bronze rows for silver.


In [0]:
%sql select * from novacart_adb.silver_schema.payments_transformed;

payment_id,order_id,payment_status,paid_amount,processed_at,bronze_ingested_at,bronze_run_id,bronze_source_table,silver_run_id,to_be_verified_by_payments_team,check_paid_amount
900001,200001,SUCCESS,100.0,2026-02-01T11:01:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900002,200002,SUCCESS,100.0,2026-02-01T11:02:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900003,200003,SUCCESS,100.0,2026-02-01T11:03:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900004,200004,SUCCESS,100.0,2026-02-01T11:04:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900005,200005,SUCCESS,100.0,2026-02-01T11:05:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900006,200006,SUCCESS,100.0,2026-02-01T11:06:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900007,200007,SUCCESS,100.0,2026-02-01T11:07:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900008,200008,SUCCESS,100.0,2026-02-01T11:08:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900009,200009,SUCCESS,100.0,2026-02-01T11:09:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False
900010,200010,SUCCESS,100.0,2026-02-01T11:10:00.000Z,2026-03-30T19:08:32.248Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.payments,598a4f4e-44dd-4a4f-9a7e-b2c97e4c9dac,No Issues,False


7.quick validation

In [0]:
print("Products transformation count:", spark.sql("select count(*) from novacart_adb.silver_schema.products_transformed").collect()[0][0])
print("orders transformation count:", spark.sql("select count(*) from novacart_adb.silver_schema.orders_transformed").collect()[0][0])
print("payments transformation count:", spark.sql("select count(*) from novacart_adb.silver_schema.payments_transformed").collect()[0][0])
display(spark.table("novacart_adb.silver_schema.products_transformed"))
display(spark.table("novacart_adb.silver_schema.orders_transformed"))
display(spark.table("novacart_adb.silver_schema.processing_control").orderBy("entity_name"))

Products transformation count: 111
orders transformation count: 0
payments transformation count: 391


product_id,product_name,category,price,updated_at,bronze_ingested_at,bronze_run_id,bronze_source_table,silver_run_id,to_be_verified_by_products_team,check_product_price
1001,PRODUCT 1,ELECTRONICS,11.0,2026-02-01T09:01:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1002,PRODUCT 2,ELECTRONICS,12.0,2026-02-01T09:02:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1003,PRODUCT 3,ELECTRONICS,13.0,2026-02-01T09:03:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1004,PRODUCT 4,ELECTRONICS,14.0,2026-02-01T09:04:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1005,PRODUCT 5,ELECTRONICS,15.0,2026-02-01T09:05:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1006,PRODUCT 6,ELECTRONICS,16.0,2026-02-01T09:06:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1007,PRODUCT 7,ELECTRONICS,17.0,2026-02-01T09:07:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1008,PRODUCT 8,ELECTRONICS,18.0,2026-02-01T09:08:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1009,PROD_9,ELECTRONICS,19.0,2026-02-01T09:09:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price
1010,PRODUCT 10,FITNESS,20.0,2026-02-01T09:10:00.000Z,2026-03-30T19:08:25.473Z,0657dba9-fdf2-467c-8184-a916c8ea5bd3,novacart_sql_connection_catalog.dbo.products,40cde124-322d-427f-94ba-e2092de27be6,No Issues,Valid_price


order_id,customer_id,product_id,order_status,order_amount,created_at,updated_at,bronze_ingested_at,bronze_run_id,bronze_source_table,silver_run_id,to_be_verified_by_orders_team,check_order_amount,order_date,order_month,order_year,order_day,order_dow


layer,entity_name,last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged,run_status,silver_run_id,upated_at
silver,orders,null,2026-03-30T19:08:17.990Z,0,success,4931629a-c8d7-4baa-a8bd-d1ad9f67b8db,2026-04-06T18:39:14.156Z
silver,payments,null,2026-03-30T19:08:32.248Z,0,success,4931629a-c8d7-4baa-a8bd-d1ad9f67b8db,2026-04-06T18:39:29.812Z
silver,products,null,2026-03-30T19:08:25.473Z,0,success,4931629a-c8d7-4baa-a8bd-d1ad9f67b8db,2026-04-06T18:39:22.962Z
